In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType, FloatType
import pyspark.sql.functions as F

catalog_name = 'ecommerce'

###Brands

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_brands")
df_bronze = df_bronze.drop("_souce_file")
display(df_bronze)


In [0]:
df_silver = df_bronze.withColumn('brand_name', F.trim(F.col('brand_name')))
display(df_silver, limit=10)

In [0]:
df_silver = df_silver.withColumn("brand_code", F.regexp_replace(F.col("brand_code"), r'[^A-Za-z0-9]', ''))
display(df_silver, limit=10)

In [0]:
df_silver.select("category_code").distinct().show(truncate=False)

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {catalog_name}.silver.slv_brands")

print("writing df to datable")
df_silver.write.format("delta") \
    .mode("overwrite") \
        .option("mergeSchema","true") \
            .saveAsTable(f"{catalog_name}.silver.slv_brands")

print("Result table slv_brands -")
display(spark.table(f"{catalog_name}.silver.slv_brands"))

###Category

In [0]:
annomalies = {
    "GROCERY" : "GRCY",
    "BOOKS": "BKS",
    "TOYS" : "TOY"
}

# df_silver = df_silver.replace(annomalies,subset="category_code")
# df_silver.select("category_code").distinct().show(truncate=False)
# display(df_silver, limit=10)

df_silver_category = spark.table(f"{catalog_name}.bronze.brz_category")
display(df_silver_category)

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_category")
df_bronze.show(10)

In [0]:
df_silver_category = df_silver_category.withColumn("category_code", F.upper(F.trim(F.col("category_code"))))

df_silver_category.select("category_code").show(truncate=False)

In [0]:
df_duplicates = df_silver_category.groupBy("category_code").count().filter(F.col("count")>1)
display(df_duplicates)

In [0]:
df_silver_category = df_silver_category.dropDuplicates(["category_code"])
display(df_silver_category)

In [0]:
df_silver_category = df_silver_category.withColumn("category_name", F.trim(F.col("category_name")))
df_silver_category.select("category_code","category_name").show(truncate=False)

In [0]:
print("Before :")
df_silver_category.printSchema()

# Drop duplicate columns with null values
df_silver_category = df_silver_category.drop("ingested_at", "_souce_file")

df_silver_category.printSchema()

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {catalog_name}.silver.slv_category")

df_silver_category.write.format("delta") \
    .mode("overwrite") \
        .option("mergeSchema","true") \
            .saveAsTable(f"{catalog_name}.silver.slv_category")

In [0]:
display(spark.table(f"{catalog_name}.silver.slv_category"))

In [0]:
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_products")

row_count, column_count = df_bronze.count(), len(df_bronze.columns)
print(f"There are {row_count} rows and {column_count} columns in the bronze table.")
display(df_bronze)

In [0]:
#replacing g with '' in weight_grams columns

df_silver = df_bronze.withColumn(
    "weight_grams",
    F.regexp_replace(F.col("weight_grams"), "g", "").cast(IntegerType()))

display(df_silver.select("weight_grams").limit(10))

In [0]:
# replace "," with "."

df_silver = df_silver.withColumn(
    "length_cm",
    F.regexp_replace(F.col("length_cm"), ",",".").cast(FloatType()))

df_silver.select("length_cm").show(10,truncate=False)

In [0]:
df_silver.select(["brand_code","category_code"]).show(10,truncate=False)

In [0]:
df_silver = df_silver.withColumn("brand_code", F.upper(F.col("brand_code"))) \
    .withColumn("category_code", F.upper(F.col("category_code")))

df_silver.select("brand_code","category_code").show(10,truncate=False)

In [0]:
df_silver.select("material").distinct().show(truncate=False)

In [0]:
# fix spelling mistakes in material column
df_silver = df_silver.withColumn("material",
                                 F.when(F.col("material") == "Coton","Cotton")
                                 .when(F.col("material") == "Alumium","Aluminum")
                                 .when(F.col("material") == "Ruber","Rubber")
                                 .otherwise(F.col("material")))

df_silver.select("material").distinct().show(truncate=False)

In [0]:
df_silver.select("rating_count").sort("rating_count").show(10,truncate=False)

In [0]:
# Convert negative rating count to positive and Null to Zero

df_silver = df_silver.withColumn("rating_count",F
                                 .when(F.col("rating_count").isNull(), 0)
                                 .when(F.col("rating_count") < 0, F.abs(F.col("rating_count")))
                                 .otherwise(F.col("rating_count")))

df_silver.select("rating_count").sort(F.desc("rating_count")).show(10,truncate=False)

In [0]:
display(df_silver)

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite").options(mergeSchema="true") \
    .saveAsTable(f"{catalog_name}.silver.slv_products")

###Customers

In [0]:
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_customers")

r, c= df_bronze.count(), len(df_bronze.columns)
print(f"r={r}, c={c}")

display(df_bronze)

In [0]:
#find rows containing null values
display(
    df_bronze.where(F.col("customer_id").isNull())
)
print(f"total null rows = {df_bronze.where(F.col("customer_id").isNull()).count()} out of {r} rows")


In [0]:
# drop all the rows containing null values
df_silver = df_bronze.dropna(subset=["customer_id"])

print(f"total null rows = {df_silver.where(F.col("customer_id").isNull()).count()} out of {df_silver.count()} rows")

In [0]:
print(f"total null rows = {df_silver.where(F.col("phone").isNull()).count()} out of {df_silver.count()} rows")

#replace "NA" with phone numbers containing null values

df_silver = df_silver.fillna("NA", subset=["phone"])

print(f"total null rows = {df_silver.where(F.col("phone").isNull()).count()} out of {df_silver.count()} rows")

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite").options(mergeSchema="true") \
    .saveAsTable(f"{catalog_name}.silver.slv_customers")

###Calendar 

In [0]:
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_calendar")

r,c = df_bronze.count(), len(df_bronze.columns)
print(f"r={r}, c={c}")

display(df_bronze)

df_bronze.printSchema()

In [0]:
from pyspark.sql.functions import to_date

df_silver = df_bronze.withColumn("date", to_date(df_bronze["date"], "dd-MM-yyyy"))

df_silver.printSchema()
display(df_silver)

In [0]:
# check duplicates

duplicates = df_silver.groupBy("date").count().filter("count > 1")
display(duplicates)
print(f"total duplicates = {duplicates.count()} out of {df_silver.count()} rows")

In [0]:
df_silver = df_silver.dropDuplicates(["date"])

current_count = df_silver.count()
print(f"Current Row count - {current_count}")

In [0]:
#Noramalize the day name
df_silver = df_silver.withColumn("day_name", F.initcap(df_silver["day_name"]))

display(df_silver)

In [0]:
# Convert Negative week of the year to positive
df_silver = df_silver.withColumn("week_of_year", F.abs(df_silver["week_of_year"]))

display(df_silver)

In [0]:
#Enhance quarter and week_of_year

df_silver = df_silver.withColumn("quarter", 
    F.concat(F.lit("Q"), F.col("quarter"), F.lit("-"), F.col("year")))

df_silver = df_silver.withColumn("week_of_year", 
    F.concat(F.lit("Week"), F.col("week_of_year"), F.lit("-"), F.col("year")))

display(df_silver)

In [0]:
df_silver = df_silver.withColumnRenamed("week_of_year","week")
display(df_silver)
df_silver.write.format("delta") \
    .mode("overwrite").options(mergeSchema="true") \
    .saveAsTable(f"{catalog_name}.silver.slv_calendar")